# Phase 3 — Absorption, Hitting Times, and the Fundamental Matrix

Companion notebook to `notes/phase3-absorption-hitting.md`. Reproduces the worked examples and the three headcount scenarios using `src.absorption`.

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd

from src.absorption import (
    absorption_probabilities,
    absorption_times,
    absorption_variances,
    canonical_form,
    first_passage_time,
    fundamental_matrix,
    hitting_probability,
    mean_return_time,
)
from src.dtmc import MarkovChain

np.set_printoptions(precision=4, suppress=True)

## 1. Worked example — three-state W/P/E chain

Working employee (transient) reaches Promoted or Exit (both absorbing) with equal weights. Reproduce $N$, $t$, and $B$.

In [ ]:
P_wpe = np.array(
    [
        [0.90, 0.05, 0.05],
        [0.00, 1.00, 0.00],
        [0.00, 0.00, 1.00],
    ]
)

print('canonical form:')
cf = canonical_form(P_wpe)
print('  Q =', cf.Q)
print('  R =', cf.R)

print('\nN =', fundamental_matrix(P_wpe))
print('t =', absorption_times(P_wpe))
print('Var(T) =', absorption_variances(P_wpe))
print('B =', absorption_probabilities(P_wpe))

## 2. Hiring freeze — full headcount with Exit absorbing

How long until each level exits, on average? Each row of $N$ shows the expected time spent at each transient level before exiting.

In [ ]:
labels_t = ['Junior', 'Mid', 'Senior']
P_freeze = np.array(
    [
        [0.93, 0.03, 0.00, 0.04],
        [0.00, 0.96, 0.02, 0.02],
        [0.00, 0.00, 0.99, 0.01],
        [0.00, 0.00, 0.00, 1.00],
    ]
)

N = fundamental_matrix(P_freeze)
df_N = pd.DataFrame(N, index=labels_t, columns=labels_t)
df_N.round(2)

In [ ]:
t = absorption_times(P_freeze)
sigma = np.sqrt(absorption_variances(P_freeze))
df_t = pd.DataFrame(
    {'E[T] (months)': t, 'std (months)': sigma},
    index=labels_t,
)
df_t.round(2)

## 3. Junior → Senior — first passage and reach probability

In [ ]:
for source, name in enumerate(labels_t):
    p = hitting_probability(P_freeze, source=source, target=2)
    print(f'P(reach Senior | start {name:<6}) = {p:.4f}')

In [ ]:
# In the absorbing variant, unconditional E[T_S] is infinite from Junior
# because there's positive probability of exiting first. The recycling
# variant gives a finite first passage time.
P_recycle = P_freeze.copy()
P_recycle[3] = [0.5, 0.0, 0.0, 0.5]
mc_recycle = MarkovChain(P_recycle, state_labels=['Junior', 'Mid', 'Senior', 'Exit'])

print(f'first_passage_time (recycling) Junior -> Senior: '
      f'{first_passage_time(mc_recycle, 0, 2):.1f} months')
print(f'first_passage_time (recycling) Mid    -> Senior: '
      f'{first_passage_time(mc_recycle, 1, 2):.1f} months')

## 4. Senior → Exit — remaining tenure

In [ ]:
# Senior absorption time = 1 / 0.01 = 100 months (geometric with p=0.01).
print(f'E[remaining tenure | Senior] = {absorption_times(P_freeze)[2]:.1f} months')
print(f'std                          = {np.sqrt(absorption_variances(P_freeze))[2]:.1f} months')

## 5. Bridge to Phase 2: mean return time = 1 / pi

On the recycling chain, the mean return time to each state is the reciprocal of its stationary probability.

In [ ]:
labels = ['Junior', 'Mid', 'Senior', 'Exit']
pi = mc_recycle.stationary()
df = pd.DataFrame(
    {
        'pi_i': pi,
        '1 / pi_i': 1 / pi,
        'mean_return_time': [mean_return_time(mc_recycle, i) for i in range(4)],
    },
    index=labels,
)
df.round(3)